## Run once for each condition
`TREATMENT = 'DMSO'` and  
`TREATMENT = 'ERL'`  

Using `EMC_model` conda environment

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import norm
import datetime
import math
import multiprocess as mp
import time
import pickle
from pysb.simulator import BngSimulator
from pysb.simulator import ScipyOdeSimulator
from scipy.signal import find_peaks
import pyarrow.feather as feather
sys.path.insert(0, "cellular_cycle")
from cell_cycle_death_model import model

import tqdm

### Set parameter values
Modify `TREATMENT` below to either `ERL` or `DMSO`.  

`CYCDI_CONC` = concentration of CDK4/6 inhibitor; set to `0` for no inhibitor.  

If using different parameter sets (e.g., those generated by PSO), change `SET-JPARAM` to `True` and choose which index value by setting `PARAM_NUM`  

To save feather files of output, set `SAVEFILE = True`

In [2]:
PARAM = pd.read_csv(Path('model_data/parameters.csv'))  # parameter file to use
INIT_CONCS = pd.read_csv(Path('model_data/initial_concentrations.csv'))  # initial concentrations for simulations

TREATMENT = 'DMSO'  # set AKT, ERK, and FAK by setting 'ERL' or 'DMSO'
 
CYCDI_CONC = 0
POINTS = 5  # number to divide each condition into, sets the number of points to divide cell cycle

# number of processes to reserve; set to either 10% or 1, whichever is greater
if round(mp.cpu_count()*0.1,0) < 1:
    RES_PROC = 1
else:
    RES_PROC = round(mp.cpu_count()*0.1,0)

PROCESSES = int(mp.cpu_count() - RES_PROC)  # number of pool processes to run at once
SAVEFILES = True  # save files for lyapunov values and mean IMT values
OVERWRITE = True  # Overwrite existing files
VERBOSE = True
SET_JPARAM = False
PARAM_NUM = 2

integrator_options = {
    "rtol": 1e-12,
    "atol": 1e-12
}

# Where to save large simulation output files
SAVE_DIR = Path(Path.home(), 'ECM_mod_output')
if not SAVE_DIR.exists():
    SAVE_DIR.mkdir()

CURDATE = datetime.datetime.now().strftime('%Y%m%d')

In [3]:
PROCESSES = 4
print(f'{PROCESSES} cores will be used in multiprocessing')

4 cores will be used in multiprocessing


In [4]:
print(f'Files to be saved in: {SAVE_DIR}')

Files to be saved in: C:\Users\olesk\ECM_mod_output


In [5]:
def run_lyaps(AKT, ERK, FAK, np=np, model=model, points=POINTS, ScipyOdeSimulator=ScipyOdeSimulator,
              find_peaks=find_peaks, integrator_options=integrator_options, norm=norm, math=math,
              time=time, verbose=VERBOSE):
    # if verbose:
    #     print(f'AKT={round(AKT,4)}, ERK={round(ERK,4)}, FAK={round(FAK,4)}, starting {time.strftime("%Y-%m-%d %H:%M:%S",time.localtime())}')

    model.parameters['AKT'].value = AKT
    model.parameters['ERK'].value = ERK
    model.parameters['FAK'].value = FAK
    ep = 0.001 * 200
    t = np.linspace(0, 1000, 1000001)
    t_0 = np.linspace(0, 2000, 2000001)
    all_logz = dict.fromkeys(list(range(1, points + 1)))

    for kk in range(points):
        # run simulation to get initial conditions (steady state values of proteins at *points* intervals)
        sim_ode = ScipyOdeSimulator(model, t_0, verbose=False, integrator_options=integrator_options).run()
        y_out = sim_ode.observables

        peaks_cdc, _ = find_peaks(y_out['obsCdc20'], prominence=0.6, distance=10)
        peaks_half = peaks_cdc[peaks_cdc < len(y_out)/2]
        peaks_half2 = peaks_cdc[peaks_cdc > len(y_out) / 2]

        if len(peaks_half2) > 3:
            IMTs = [t_0[y] - t_0[x] for x, y in zip(peaks_half2, peaks_half2[1:])]
            meanIMT = sum(IMTs) / len(IMTs)
            variable_range = y_out[peaks_half[-3]:peaks_half[-2]]
            tmp_loc = (round(len(variable_range) / (points)) * (kk))
        else:
            meanIMT = 0
            variable_range = y_out[575000:600000]
            tmp_loc = (round(len(variable_range) / (points)) * (kk))
        if verbose:
            print(f'AKT={AKT}, ERK={ERK}, FAK={FAK}, meanIMT={meanIMT}')


        Initial_1 = variable_range[tmp_loc]
        Initial_1 = [val + ep for val in Initial_1]

        prots = ['Skp2_0', 'p21_0', 'p21p_0', 'cdh1act_0', 'cdh1inact_0', 'Myc_0', 'CycD_0', 'CycDp21_0', 'pRb_0',
                 'pRbp_0', 'E2F_0', 'pRbE2F_0', 'CycE_0', 'CycEp21_0', 'CycA_0', 'CycAp21_0', 'CycB_0', 'CycBp21_0',
                 'Cdc20_0']
        prot_init = {prots[i]: Initial_1[i] for i in range(len(prots))}

        # set initial conditions from first simulation
        for key in prot_init.keys():
            model.parameters[key].value = prot_init[key] / model.parameters['omega'].value

        sim_ode1 = ScipyOdeSimulator(model, t, verbose=False, integrator_options=integrator_options).run()
        y_out1 = sim_ode1.observables

        z = [0] * (round((len(y_out1) / 2000)) + 1)
        n = 0
        for i in range(0, len(y_out1), 2000):
            if len(peaks_half2) > 0:
                window = y_out[peaks_half[-3]+i+tmp_loc-5000:peaks_half[-2]+i+tmp_loc+5000]
            else:
                window = y_out[575000+i+tmp_loc-5000:600000+i+tmp_loc+5000]

            dists = [0] * len(window)
            for j in range(0, len(window)):
                dif = np.asarray(window[j].tolist())-np.asarray(y_out1[i].tolist())
                dists[j] = norm(dif[0:19]/200)

            z[n] = min(dists)
            n = n + 1

        logz = [math.log(num) for num in z]
        all_logz[kk + 1] = logz
        # if verbose:
        #     print(f'AKT={round(AKT,4)}, ERK={round(ERK,4)}, FAK={round(FAK,4)}, kk={kk}, finished {time.strftime("%Y-%m-%d %H:%M:%S",time.localtime())}')
        if verbose:
            print([AKT, ERK, FAK, kk])

    return all_logz, meanIMT

In [6]:
t = np.linspace(0, 1000, 1000001)


In [7]:
# Importing the parameters:

for key in PARAM.keys():
    try:
        if model.parameters[key].value != PARAM[key][0]:
            print(f'Replacing {key} with {PARAM[key]}')
            model.parameters[key].value = PARAM[key]
    except:
        print(f'{key} not found in model')
model.parameters['omega'].value = 200


if SET_JPARAM:
    with open('fit_data/pso/frac_prolif_parameters_pso_sorted.pkl', 'rb') as f:
        frac_prolif_parameters_pso = pickle.load(f)
    # new_param_pso_sort = frac_prolif_parameters_pso.sort_values([('cost', 'total cost likelihood')])
    new_params = frac_prolif_parameters_pso['model parameters']
    # Importing the parameters:
    for key in new_params.keys():
        try:
            if model.parameters[key].value != new_params[key].iloc[PARAM_NUM]:
                print(f'Replacing {key} with {new_params[key].iloc[PARAM_NUM]}')
                model.parameters[key].value = new_params[key].iloc[PARAM_NUM]
        except:
            print(f'{key} not found in model')
    model.parameters['omega'].value = 200

for key in INIT_CONCS.keys():
    try:
        model.parameters[key].value = INIT_CONCS[key][0] / model.parameters['omega'].value
    except:
        print(f'{key} not found in model initial concentrations')

model.parameters['CycDi_0'].value = CYCDI_CONC

if TREATMENT == 'ERL':
    FAKs = list(np.linspace(0.4,0.6,50))
    tmp = list(np.linspace(0.6,1,50))
    FAKs.extend(tmp)
    AKTs = list(np.linspace(0.3,0.6,50))
    tmp = [0.6]*50
    AKTs.extend(tmp)
    ERKs = [0.2]*100
else:
    FAKs = list(np.linspace(0.4,0.6,50))
    tmp = list(np.linspace(0.6,1,50))
    FAKs.extend(tmp)
    AKTs = list(np.linspace(0.6,1,50))
    tmp = [1]*50
    AKTs.extend(tmp)
    ERKs = [1]*100

conds = [(AKT, ERK, FAK) for AKT, ERK, FAK, in zip(AKTs, ERKs, FAKs)]

In [8]:
conds[0]

(0.6, 1, 0.4)

In [ ]:
start = time.localtime()
print(f'Start time is {time.strftime("%Y-%m-%d %H:%M:%S",start)}')
results = []

with mp.Pool(processes = PROCESSES) as p:
    results = p.starmap(run_lyaps, conds)
    # for result in p.starmap(run_lyaps, conds):
    #     print(f'Appending result at {time.strftime("%Y-%m-%d %H:%M:%S",time.localtime())}')
    #     results.append(result)

end = time.localtime()
print(f'End time is {time.strftime("%Y-%m-%d %H:%M:%S",end)}')
print(f'Process took {np.round((time.mktime(end) - time.mktime(start))/60, 2)} minutes')

Start time is 2025-07-01 17:50:06


In [9]:
assert len(results) == len(conds)

In [10]:
z_list = [num[0] for num in results]
mean_IMT = [num[1] for num in results]
len(mean_IMT)

100

In [11]:
z_list = [num[0] for num in results]
mean_IMT = [num[1] for num in results]

df_lyap = pd.DataFrame(z_list)
df_meanIMT = pd.DataFrame(mean_IMT)

cycdi = str(round(model.parameters['CycDi_0'].value % 1,2)).split('.')[1]

In [12]:
if SET_JPARAM:
    lyap_filename = Path(SAVE_DIR, f'lyaps_{TREATMENT}_pso_{PARAM_NUM}_cycdi_0_{cycdi}.fea') # feather might have an issue with Path object
else:
    lyap_filename = Path(SAVE_DIR, f'lyaps_{TREATMENT}_cycdi_0_{cycdi}.fea')

if (not lyap_filename.exists() or OVERWRITE) and SAVEFILES:
    print(f'Saving file to {lyap_filename}...')
    feather.write_feather(df_lyap, lyap_filename)
else:
    print(f'File {lyap_filename} will not be saved.')

Saving file to /Users/darren/ECM_mod_output/lyaps_DMSO_pso_2_cycdi_0_0.fea...


In [13]:
if SET_JPARAM:
    meanIMT_filename = Path(SAVE_DIR, f'mINT_{TREATMENT}_pso_{PARAM_NUM}_cycdi_0_{cycdi}.fea')
else:
    meanIMT_filename = Path(SAVE_DIR, f'mINT_{TREATMENT}_cycdi_0_{cycdi}.fea')

if (not meanIMT_filename.exists() or OVERWRITE) and SAVEFILES:
    print(f'Saving file to {meanIMT_filename}...')
    feather.write_feather(df_meanIMT, meanIMT_filename)
else:
    print(f'File {meanIMT_filename} will not be saved.')

Saving file to /Users/darren/ECM_mod_output/mINT_DMSO_pso_2_cycdi_0_0.fea...
